3.1

In [ ]:
R, C = map(int, input().split())
parking_map = [input().strip() for _ in range(R)]


In [14]:
R, C = 4, 4
parking_map = [
    "#..#",
    "..X.",
    "..X.",
    "#XX#",
]

In [16]:

counts = [0] * 5
for i in range(R - 1):
    for j in range(C - 1):
        block = [parking_map[i][j], parking_map[i][j+1], parking_map[i+1][j], parking_map[i+1][j+1]]
        if '#' in block:
            continue
        counts[block.count('X')] += 1

for count in counts:
    print(count)


1
1
2
1
0


In [ ]:
import bisect

def closest_sum(L, queries):
    L.sort()

    def find_closest_sum(query):
        closest_sum = None
        closest_dist = float('inf')

        for i in range(len(L)):
            target = query - L[i]
            pos = bisect.bisect_left(L, target, i + 1) 
            
            for j in [pos, pos - 1]:
                if j < len(L) and j != i:  
                    current_sum = L[i] + L[j]
                    current_dist = abs(current_sum - query)
                    
                    if current_dist < closest_dist:
                        closest_sum = current_sum
                        closest_dist = current_dist
                    elif current_dist == closest_dist:
                        closest_sum = min(closest_sum, current_sum)
                        
        return closest_sum

    results = []
    for query in queries:
        closest = find_closest_sum(query)
        results.append(f"Closest sum to {query} is {closest}.")

    return "\n".join(results)

def main():
    case_number = 1
    while True:
        n = int(input())
        if n == 0:
            break 

        L = [int(input()) for _ in range(n)]

        m = int(input())

        queries = [int(input()) for _ in range(m)]

        print(f"Case {case_number}:")
        print(closest_sum(L, queries))

        case_number += 1

main()


Case 1:
Closest sum to 1 is 15.
Closest sum to 51 is 51.
Closest sum to 30 is 29.


In [25]:
def reversed_splits(s):
    for i in range(1, len(s) - 1):
        for j in range(i + 1, len(s)):
            yield s[0:i][::-1] + s[i:j][::-1] + s[j:][::-1]

def main():
    s = input()
    
    return min(reversed_splits(s))

main()

'bometil'

In [ ]:
def interpolate(a_prev_h, a_prev_t, a_delta_h, a_delta_t, d_prev_h, d_prev_t, d_delta_h, d_delta_t):
    A = a_delta_h / a_delta_t
    B = d_delta_h / d_delta_t
    return (A * a_prev_t - a_prev_h + d_prev_h + B * d_prev_t) / (A + B)

def main():
    a, d = map(int, input().split())
    ascent = [tuple(map(int, input().split())) for _ in range(a)]
    descent = [tuple(map(int, input().split())) for _ in range(d)]
    
    total_descent_h = sum(h for h, _ in descent)
    
    f = (0, 0)
    g = (total_descent_h, 0)
    
    for i in range(min(a, d)):
        f_prev = f
        g_prev = g
        
        a_dh, a_dt = ascent[i]
        d_dh, d_dt = descent[i]
        
        f = (f[0] + a_dh, f[1] + a_dt)
        g = (g[0] - d_dh, g[1] + d_dt)
        
        if f[0] > g[0]:
            t = interpolate(f_prev[0], f_prev[1], a_dh, a_dt, g_prev[0], g_prev[1], d_dh, d_dt)
print(f"{t:.6f}")
    
main()


5.238095


In [ ]:
import heapq

def nikola(n, fees):
    MAX = sum(fees) + 1
    dist = [[MAX] * (n+2) for _ in range(n+2)]
    hq = []
    dist[2][1] = fees[1]
    heapq.heappush(hq, (fees[1], 2, 1))

    while hq:
        cost, pos, jump = heapq.heappop(hq)
        if cost > dist[pos][jump]:
            continue
        nxt = pos + jump + 1
        if nxt <= n:
            nc = cost + fees[nxt-1]
            if nc < dist[nxt][jump+1]:
                dist[nxt][jump+1] = nc
                heapq.heappush(hq, (nc, nxt, jump+1))
        prv = pos - jump
        if prv >= 1:
            nc = cost + fees[prv-1]
            if nc < dist[prv][jump]:
                dist[prv][jump] = nc
                heapq.heappush(hq, (nc, prv, jump))
    return min(dist[n][1:n+1])

N = int(input())
fees = [int(input()) for _ in range(N)]
print(nikola(N, fees))


12
14


In [ ]:
def main():
    n = int(input())
    costs = list(map(int, input().split()))
    m = int(input())
    orders = list(map(int, input().split()))
    max_s = max(orders)

    dp = [-2] * (max_s + 1)
    dp[0] = 0

    for i, c in enumerate(costs, start=1):
        for s in range(0, max_s - c + 1):
            if dp[s] == -2:
                continue

            tgt = s + c

            if dp[s] == -1:
                dp[tgt] = -1
            else:
                # source was unique
                if dp[tgt] == -2:
                    dp[tgt] = i
                else:
                    dp[tgt] = -1

    out = []
    for S in orders:
        if S > max_s or dp[S] == -2:
            out.append("Impossible")
        elif dp[S] == -1:
            out.append("Ambiguous")
        else:
            seq = []
            cur = S
            while cur > 0:
                item = dp[cur]
                seq.append(item)
                cur -= costs[item - 1]
            seq.sort()
            out.append(" ".join(map(str, seq)))

    print("\n".join(out))


if __name__ == "__main__":
    main()


Impossible
Ambiguous
1 2 2


In [4]:
def best_walrus_weight(plates):
    TARGET = 1000
    MAX_PLATE = max(plates) if plates else 0
    MAX_SUM = TARGET + MAX_PLATE  # we only care up to this
    
    # reachable[s] = can we pick some subset to sum exactly to s?
    reachable = [False] * (MAX_SUM + 1)
    reachable[0] = True
    
    for w in plates:
        # unbounded knapsack: you can use each plate at most once, so iterate backwards
        for s in range(MAX_SUM - w, -1, -1):
            if reachable[s]:
                reachable[s + w] = True
    
    # now scan outwards from TARGET: at distance d,
    # prefer TARGET+d over TARGET-d
    for d in range(0, TARGET + 1):
        hi = TARGET + d
        if hi <= MAX_SUM and reachable[hi]:
            return hi
        lo = TARGET - d
        if lo >= 0 and reachable[lo]:
            return lo

# --- I/O wrapper ---
if __name__ == "__main__":
    
    n = int(input().strip())
    plates = [int(input()) for _ in range(n)]
    ans = best_walrus_weight(plates)
    print(ans)


1


In [ ]:
import bisect

def min_nested_dolls(dolls):
    # dolls: list of (w,h)
    dolls.sort(key=lambda x: (x[0], -x[1]))
    # extract heights and negate
    negH = [-h for _,h in dolls]

    # patience‐sorting piles for LIS on negH
    piles = []
    for x in negH:
        # find first pile whose top >= x
        i = bisect.bisect_left(piles, x)
        if i == len(piles):
            piles.append(x)
        else:
            piles[i] = x
    return len(piles)


def main():
    t = int(input())
    for _ in range(t):
        m = int(input())
        data = list(map(int, input().split()))
        dolls = [(data[i], data[i + 1]) for i in range(0, 2 * m, 2)]
        print(min_nested_dolls(dolls))


if __name__ == "__main__":
    main()


2


In [ ]:
def process_store(lines):
    T = 1440
    B = [-0.08] * T
    for line in lines:
        t, p = line
        B[t] += p

    P = [0.0] * (T + 1)
    for i in range(T):
        P[i+1] = P[i] + B[i]

    min_prefix = 0.0
    min_index  = 0
    best_profit = -1e18
    best_start = 0
    best_end = -1
    best_length = T + 1

    for end in range(T):
        curr = P[end+1] - min_prefix
        length = end - min_index + 1

        if (curr > best_profit
            or (abs(curr - best_profit) < 1e-12 and length < best_length)
            or (abs(curr - best_profit) < 1e-12 and length == best_length and min_index < best_start)
        ):
            best_profit = curr
            best_start = min_index
            best_end = end
            best_length = length

        if P[end+1] < min_prefix - 1e-12:
            min_prefix = P[end+1]
            min_index = end + 1
        elif abs(P[end+1] - min_prefix) < 1e-12:
            min_index = end + 1

    if best_profit <= 1e-12:
        return "no profit"
    
    return f"{best_profit:.2f} {best_start} {best_end}"

def main():
    n=None
    while n!=0:
        n = int(input().strip())
        lines = []
        for _ in range(n):
            t, p = input().split()
            lines.append((int(t), float(p)))
        print(process_store(lines))

if __name__ == "__main__":
    main()


In [ ]:
import sys

def main():
    input = sys.stdin.readline
    N, M = map(int, input().split())

    g = [[] for _ in range(N+1)]
    for _ in range(M):
        a, b = map(int, input().split())
        g[a].append(b)
        g[b].append(a)

    connected = [False] * (N+1)
    stack = [1]
    connected[1] = True

    while stack:
        u = stack.pop()
        for v in g[u]:
            if not connected[v]:
                connected[v] = True
                stack.append(v)


    # collect all unconnected houses
    unconnected = [str(i) for i in range(1, N+1) if not connected[i]]

    if not unconnected:
        print("Connected")
    else:
        print("\n".join(unconnected))

if __name__ == "__main__":
    main()
